<a href="https://colab.research.google.com/github/burakderee/siir-olusturucu/blob/main/llama_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install unsloth bayesian-optimization nltk -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
print("✅ Model yüklendi.")

In [ ]:
from datasets import load_dataset, concatenate_datasets

orijinal = load_dataset("json", data_files="/content/drive/MyDrive/orhan_veli_dogru_format.jsonl")["train"]
# v2 → v3
sentetik = load_dataset("json", data_files="/content/drive/MyDrive/orhan_veli_sentetik_v3.jsonl")["train"]
dataset_raw = concatenate_datasets([orijinal, sentetik])

# %80 train, %20 validation
split = dataset_raw.train_test_split(test_size=0.2, seed=42)

SABLON = """### Talimat:
Sen Orhan Veli Kanık'sın. Garip akımı üslubunda, kafiyesiz, sade bir şiir yaz.
ZORUNLU: "{input}" kelimesi şiirde geçmeli.

### Anahtar Kelime:
{input}

### Şiir:
{output}<|end_of_text|>"""

train_fmt = split["train"].map(lambda x: {"text": SABLON.format(input=x["input"], output=x["output"])})
val_fmt   = split["test"].map(lambda x: {"text": SABLON.format(input=x["input"], output=x["output"])})

print(f"✅ Train: {len(train_fmt)} | Validation: {len(val_fmt)}")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("✅ LoRA hazır.")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback
import copy

class ManuelEarlyStopping(TrainerCallback):
    def __init__(self, patience=2, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.en_iyi_val_loss = float('inf')
        self.bekleyis = 0

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        val_loss = metrics.get("eval_loss", float('inf'))
        print(f"Epoch {state.epoch:.0f} | Val: {val_loss:.4f} | Best: {self.en_iyi_val_loss:.4f}")
        if val_loss < self.en_iyi_val_loss - self.min_delta:
            self.en_iyi_val_loss = val_loss
            self.bekleyis = 0
            print("✅ Yeni en iyi model!")
        else:
            self.bekleyis += 1
            print(f"⏳ İyileşme yok ({self.bekleyis}/{self.patience})")
            if self.bekleyis >= self.patience:
                print("🛑 Early stopping!")
                control.should_training_stop = True

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_fmt,
    eval_dataset=val_fmt,
    dataset_text_field="text",
    max_seq_length=512,
    args=TrainingArguments(
        num_train_epochs=10,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=5e-6,
        lr_scheduler_type="cosine",
        warmup_steps=124,
        weight_decay=0.05,
        bf16=True,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",          # ← checkpoint kapatıldı
        load_best_model_at_end=False, # ← kapatıldı
        output_dir="./llama_output",
    ),
    callbacks=[ManuelEarlyStopping(patience=2, min_delta=0.001)]
)

print("🚀 Eğitim başlıyor...")
sonuc = trainer.train()
print(f"✅ Bitti. Train Loss: {sonuc.training_loss:.4f}")

In [ ]:
model.save_pretrained("/content/drive/MyDrive/llama3_orhan_veli_v2")
tokenizer.save_pretrained("/content/drive/MyDrive/llama3_orhan_veli_v2")
print("✅ Model kaydedildi.")

In [ ]:
# Adapter'ı temizle, base modele dön
from peft import PeftModel
import torch

if isinstance(model, PeftModel):
    model = model.merge_and_unload()
    print("✅ Adapter temizlendi, base model hazır.")

torch.cuda.empty_cache()

In [ ]:
from datasets import load_dataset, concatenate_datasets
from trl import SFTTrainer
from transformers import TrainingArguments
import json, torch

# Veri seti — hücre içinde tanımla
orijinal = load_dataset("json", data_files="/content/drive/MyDrive/orhan_veli_dogru_format.jsonl")["train"]
sentetik  = load_dataset("json", data_files="/content/drive/MyDrive/orhan_veli_sentetik_v3.jsonl")["train"]
dataset_raw = concatenate_datasets([orijinal, sentetik])

SABLON = """### Talimat:
Sen Orhan Veli Kanık'sın. Garip akımı üslubunda, kafiyesiz, sade bir şiir yaz.
ZORUNLU: "{input}" kelimesi şiirde geçmeli.

### Anahtar Kelime:
{input}

### Şiir:
{output}<|end_of_text|>"""

dataset_fmt = dataset_raw.map(lambda x: {"text": SABLON.format(input=x["input"], output=x["output"])})
print(f"✅ {len(dataset_fmt)} örnek hazır.")

# Grid Search
DENEYLER = [
    {"isim": "A", "lr": 1e-4, "epoch": 2, "r": 8},
    {"isim": "B", "lr": 2e-4, "epoch": 3, "r": 16},
    {"isim": "C", "lr": 1e-4, "epoch": 3, "r": 32},
]

llama_sonuclar = []

for deney in DENEYLER:
    print(f"\n{'='*50}")
    print(f"DENEY {deney['isim']} — lr={deney['lr']} epoch={deney['epoch']} r={deney['r']}")
    print('='*50)

    torch.cuda.empty_cache()

    model_peft = FastLanguageModel.get_peft_model(
        model,
        r=deney["r"],
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        lora_alpha=deney["r"]*2,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    trainer = SFTTrainer(
        model=model_peft,
        tokenizer=tokenizer,
        train_dataset=dataset_fmt,
        dataset_text_field="text",
        max_seq_length=1024,
        args=TrainingArguments(
            num_train_epochs=deney["epoch"],
            per_device_train_batch_size=4,
            gradient_accumulation_steps=2,
            learning_rate=deney["lr"],
            lr_scheduler_type="cosine",
            warmup_steps=5,
            weight_decay=0.01,
            bf16=True,
            logging_steps=999,
            output_dir="./tmp",
            save_strategy="no",
        ),
    )

    sonuc = trainer.train()
    son_loss = sonuc.training_loss

    llama_sonuclar.append({
        "model": "llama-3-8b",
        "deney": deney["isim"],
        "lr": deney["lr"],
        "epoch": deney["epoch"],
        "r": deney["r"],
        "loss": son_loss
    })

    print(f"✅ Deney {deney['isim']} — Loss: {son_loss:.4f}")

    # Adapter sıfırla
    for name, param in model_peft.named_parameters():
        if "lora" in name.lower():
            param.data.zero_()
    torch.cuda.empty_cache()

# Kaydet
with open("/content/drive/MyDrive/llama_grid_sonuclari.json", "w") as f:
    json.dump(llama_sonuclar, f, indent=2)

en_iyi = min(llama_sonuclar, key=lambda x: x["loss"])
print(f"\n{'='*50}")
print("LLAMA GRID SEARCH SONUCU")
print('='*50)
for s in llama_sonuclar:
    print(f"  Deney {s['deney']}: lr={s['lr']} epoch={s['epoch']} r={s['r']} → Loss={s['loss']:.4f}")
print(f"\n🏆 En iyi: Deney {en_iyi['deney']} — Loss={en_iyi['loss']:.4f}")

In [ ]:
!pip install bayesian-optimization

import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset, concatenate_datasets
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback
from unsloth import FastLanguageModel
import json, torch
from peft import PeftModel

# ── DATASET HAZIRLIĞI ──────────────────────────────────────
orijinal = load_dataset("json", data_files="/content/drive/MyDrive/orhan_veli_dogru_format.jsonl")["train"]
sentetik  = load_dataset("json", data_files="/content/drive/MyDrive/orhan_veli_sentetik_v3.jsonl")["train"]
dataset_raw = concatenate_datasets([orijinal, sentetik])

SABLON = """### Talimat:
Sen Orhan Veli Kanık'sın. Garip akımı üslubunda, kafiyesiz, sade bir şiir yaz.
ZORUNLU: "{input}" kelimesi şiirde geçmeli.

### Anahtar Kelime:
{input}

### Şiir:
{output}<|end_of_text|>"""

dataset_fmt = dataset_raw.map(lambda x: {"text": SABLON.format(input=x["input"], output=x["output"])})

# ── 1. OVERFİTTİNG ANALİZİ ──────────────────────────────────
DENEYLER = [
    {"isim": "A", "lr": 1e-4, "epoch": 2, "r": 8,  "renk": "blue"},
    {"isim": "B", "lr": 2e-4, "epoch": 3, "r": 16, "renk": "green"},
    {"isim": "C", "lr": 1e-4, "epoch": 3, "r": 32, "renk": "red"},
]

overfitting_verileri = {}

for deney in DENEYLER:
    print(f"\nDeney {deney['isim']} başlatılıyor...")
    torch.cuda.empty_cache()

    if isinstance(model, PeftModel):
        model = model.merge_and_unload()
    torch.cuda.empty_cache()

    model_peft = FastLanguageModel.get_peft_model(
        model,
        r=deney["r"],
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        lora_alpha=deney["r"]*2,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    epoch_kayiplari = []

    class EpochLossCallback(TrainerCallback):
        def on_epoch_end(self, args, state, control, **kwargs):
            son_loglar = [l for l in state.log_history if "loss" in l]
            if son_loglar:
                epoch_kayiplari.append(son_loglar[-1]["loss"])

    trainer = SFTTrainer(
        model=model_peft,
        tokenizer=tokenizer,
        train_dataset=dataset_fmt,
        dataset_text_field="text",
        max_seq_length=1024,
        args=TrainingArguments(
            num_train_epochs=deney["epoch"],
            per_device_train_batch_size=4,   # 1→4
            gradient_accumulation_steps=2,   # 8→2
            learning_rate=deney["lr"],
            lr_scheduler_type="cosine",
            warmup_steps=5,
            weight_decay=0.01,
            bf16=True,                       # fp16→bf16
            fp16=False,
            logging_steps=1,
            output_dir="./tmp",
            save_strategy="no",
        ),
        callbacks=[EpochLossCallback()]
    )

    trainer.train()

    while len(epoch_kayiplari) < deney["epoch"]:
        epoch_kayiplari.append(trainer.state.log_history[-1].get("loss", 0))

    overfitting_verileri[deney["isim"]] = {
        "kayiplar": epoch_kayiplari[:deney["epoch"]],
        "renk": deney["renk"],
        "lr": deney["lr"],
        "epoch": deney["epoch"],
        "r": deney["r"]
    }

# Grafik
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (isim, veri) in enumerate(overfitting_verileri.items()):
    kayiplar = veri["kayiplar"]
    ax = axes[i]
    ax.plot(range(1, len(kayiplar)+1), kayiplar,
            color=veri["renk"], marker='o', linewidth=2, markersize=6)

    if len(kayiplar) >= 3:
        son_uc = kayiplar[-3:]
        if son_uc[-1] > son_uc[0]:
            ax.axvspan(len(kayiplar)-1, len(kayiplar),
                      alpha=0.2, color='red', label='Overfitting şüphesi')

    en_iyi_idx = np.argmin(kayiplar)
    ax.scatter(en_iyi_idx+1, kayiplar[en_iyi_idx],
               color='gold', s=150, zorder=5, marker='*',
               label=f"En iyi: {kayiplar[en_iyi_idx]:.4f}")
    ax.set_title(f"Deney {isim}\nlr={veri['lr']} epoch={veri['epoch']} r={veri['r']}",
                 fontsize=10, fontweight='bold')
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Training Loss")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle("LLaMA-3 8B — Overfitting Analizi", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/llama_overfitting_analizi_v2.png", dpi=150)
plt.show()

# ── 2. BAYESIAN OPTİMİZASYON ────────────────────────────────
from bayes_opt import BayesianOptimization

bayesian_sonuclar = []

def llama_egit_bayes(learning_rate, epoch):
    global model
    epoch = int(round(epoch))
    r_sabit = 16

    torch.cuda.empty_cache()
    if isinstance(model, PeftModel):
        model = model.merge_and_unload()
    torch.cuda.empty_cache()

    model_peft = FastLanguageModel.get_peft_model(
        model,
        r=r_sabit,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        lora_alpha=r_sabit*2,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    trainer = SFTTrainer(
        model=model_peft,
        tokenizer=tokenizer,
        train_dataset=dataset_fmt,
        dataset_text_field="text",
        max_seq_length=1024,
        args=TrainingArguments(
            num_train_epochs=epoch,
            per_device_train_batch_size=4,   # 1→4
            gradient_accumulation_steps=2,   # 8→2
            learning_rate=learning_rate,
            lr_scheduler_type="cosine",
            warmup_steps=5,
            weight_decay=0.01,
            bf16=True,                       # fp16→bf16
            fp16=False,
            logging_steps=10,
            output_dir="./tmp",
            save_strategy="no",
        ),
    )

    sonuc = trainer.train()
    son_loss = sonuc.training_loss

    bayesian_sonuclar.append({
        "lr": learning_rate,
        "epoch": epoch,
        "r": r_sabit,
        "loss": son_loss
    })

    print(f"  lr={learning_rate:.6f} epoch={epoch} → Loss={son_loss:.4f}")
    return -son_loss

pbounds = {
    "learning_rate": (5e-5, 3e-4),
    "epoch": (3, 5),
}

optimizer_bayes = BayesianOptimization(
    f=llama_egit_bayes,
    pbounds=pbounds,
    random_state=42,
    verbose=0
)

print("\n🔍 LLaMA Bayesian Optimizasyon başlıyor...\n")
optimizer_bayes.maximize(init_points=2, n_iter=2)  # 5→4 iterasyon

en_iyi = min(bayesian_sonuclar, key=lambda x: x["loss"])
print(f"\n{'='*50}")
print("LLAMA BAYESIAN SONUCU")
print('='*50)
for s in bayesian_sonuclar:
    print(f"  lr={s['lr']:.6f} epoch={s['epoch']} → Loss={s['loss']:.4f}")
print(f"\n🏆 En iyi: lr={en_iyi['lr']:.6f} epoch={en_iyi['epoch']} → Loss={en_iyi['loss']:.4f}")

import json
with open("/content/drive/MyDrive/llama_bayesian_v2_sonuclari.json", "w") as f:
    json.dump({
        "grid": [
            {"deney": "A", "lr": 1e-4, "epoch": 2, "r": 8},
            {"deney": "B", "lr": 2e-4, "epoch": 3, "r": 16},
            {"deney": "C", "lr": 1e-4, "epoch": 3, "r": 32},
        ],
        "bayesian": bayesian_sonuclar,
        "en_iyi": en_iyi
    }, f, indent=2)

print("✅ Sonuçlar kaydedildi.")

In [ ]:
!pip install nltk -q

import torch, math, json, re, numpy as np, nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from unsloth import FastLanguageModel
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

TEST_KELIMELERI = [
    "wifi", "suşi", "selfie", "bitcoin", "çiğköfte",
    "kargo", "şarj", "deadline", "metro", "market",
    "tiktok", "barista", "influencer", "netflix", "sipariş",
    "asansör", "navigasyon", "zihin", "iftira", "trip"
]

REFERANS_SIIRLER = [
    "sokakta yürürüm ben de insanlar gibi",
    "ne güzel şeydir yaşamak bu dünyada",
    "istanbul'u dinliyorum gözlerim kapalı",
    "garip benim garip istanbul garip hayat",
    "para yok cebimde ne yapayım"
]

def siir_uret(model, tokenizer, kelime, max_deneme=3):
    prompt = """### Talimat:
Sen Orhan Veli Kanık'sın. Garip akımı üslubunda, kafiyesiz, sade bir şiir yaz.
YASAK: kafiye, dramatik dil.
ZORUNLU: "{kelime}" kelimesi şiirde geçmeli.

### Anahtar Kelime:
{kelime}

### Şiir (içinde "{kelime}" geçmeli):
""".format(kelime=kelime)

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    for _ in range(max_deneme):
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.8,
                top_p=0.9,
                top_k=50,
                repetition_penalty=1.3,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        prompt_len = inputs["input_ids"].shape[1]
        siir = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
        dizeler = [d for d in siir.split('\n') if d.strip()]
        siir = '\n'.join(dizeler[:8])
        if kelime.lower() in siir.lower():
            return siir
    return siir

def kelime_kullanim_orani(siirler, kelimeler):
    return sum(1 for k, s in zip(kelimeler, siirler) if k.lower() in s.lower()) / len(kelimeler)

def perplexity_hesapla(model, tokenizer, metinler):
    model.eval()
    toplam, sayi = 0, 0
    for metin in metinler:
        enc = tokenizer(metin, return_tensors="pt", truncation=True, max_length=512).to("cuda")
        with torch.no_grad():
            cikti = model(**enc, labels=enc["input_ids"])
            toplam += cikti.loss.item()
            sayi += 1
    return math.exp(toplam / sayi)

def bleu_hesapla(siirler, referanslar):
    smoother = SmoothingFunction().method1
    skorlar = []
    for siir in siirler:
        hip = nltk.word_tokenize(siir.lower())
        refs = [nltk.word_tokenize(r.lower()) for r in referanslar]
        skorlar.append(sentence_bleu(refs, hip, smoothing_function=smoother))
    return np.mean(skorlar)

# LLaMA zaten bellekte — adapter yüklü mü kontrol et
FastLanguageModel.for_inference(model)
print("✅ LLaMA inference modunda.")

print("\n📊 LLaMA metrikleri hesaplanıyor...")

llama_siirler = []
for kelime in TEST_KELIMELERI:
    siir = siir_uret(model, tokenizer, kelime)
    llama_siirler.append(siir)
    durum = "✅" if kelime.lower() in siir.lower() else "❌"
    print(f"  {durum} {kelime}")

kko  = kelime_kullanim_orani(llama_siirler, TEST_KELIMELERI)
perp = perplexity_hesapla(model, tokenizer, llama_siirler)
bleu = bleu_hesapla(llama_siirler, REFERANS_SIIRLER)

llama_metrikler = {
    "model": "LLaMA-3 8B",
    "kelime_kullanim_orani": round(kko, 4),
    "perplexity": round(perp, 2),
    "bleu": round(bleu, 4),
    "en_iyi_loss": 1.1181
}

print(f"\n{'='*45}")
print(f"  LLAMA-3 8B METRİKLERİ")
print(f"{'='*45}")
print(f"  Kelime Kullanım Oranı : %{kko*100:.1f}")
print(f"  Perplexity            : {perp:.2f}")
print(f"  BLEU Skoru            : {bleu:.4f}")
print(f"{'='*45}")

with open("/content/drive/MyDrive/llama_metrikler.json", "w") as f:
    json.dump(llama_metrikler, f, indent=2, ensure_ascii=False)

print("✅ LLaMA metrikleri kaydedildi.")